# 08 - WEO Excel SciPy Statistics Lab

This lab uses the IMF World Economic Outlook Excel workbook from Module 4. You will create clean analysis DataFrames from multiple sheets, then run descriptive statistics, confidence intervals, correlations, outlier checks, and hypothesis tests with SciPy.

## Important: Use a Python Notebook Runtime

Run this notebook with a **Python 3** kernel in Jupyter, VS Code, JupyterLab, or Google Colab. Do not run these cells in SQL Server query mode.

This notebook uses the WEO Excel workbook from Module 4. Locally, it looks for `Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx`. In Google Colab, it also checks `/content/WEOApr2026all.xlsx` and can prompt you to upload the file if needed.

## 0. Install Packages

This cell installs the packages needed for this notebook. `%pip` is a notebook command, not SQL syntax. If a package was just updated, Jupyter or Colab may ask you to restart the kernel/runtime before continuing.

In [40]:
%pip install pandas numpy scipy matplotlib seaborn scikit-learn openpyxl joblib -q

Note: you may need to restart the kernel to use updated packages.


## 1. Imports

Imports load Python libraries into memory. Keeping imports in their own cell makes it easy to see what tools the notebook uses.

In [41]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
from scipy import stats

## 2. Display and Output Settings

`pd.set_option("display.max_columns", 60)` tells pandas to show up to 60 columns when displaying a table.

`pd.set_option("display.width", 160)` gives pandas more horizontal space before wrapping printed output.

These settings only affect how tables look in the notebook. They do not change the dataset.

In [42]:
# These settings only control how pandas tables are displayed in the notebook.
# They do not change the data.
pd.set_option("display.max_columns", 60)  # Show up to 60 columns before pandas hides columns with "...".
pd.set_option("display.width", 160)       # Make printed tables wider so rows wrap less often.

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid") if "sns" in globals() else None

## 3. Locate the WEO Workbook

This function checks several possible file paths. The `/content/...` path is used by Google Colab when a file is uploaded to the session.

In [43]:
def find_weo_workbook():
    """Find the WEO workbook locally, in Colab, or through a Colab upload prompt."""
    candidate_paths = [
        Path("../../../Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("data/WEOApr2026all.xlsx"),
        Path("/content/WEOApr2026all.xlsx"),
        Path("WEOApr2026all.xlsx"),
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    # This block only runs in Google Colab. It lets a learner upload the Excel file manually.
    try:
        from google.colab import files
        print("Upload WEOApr2026all.xlsx from Module 4 to continue.")
        uploaded = files.upload()
        if "WEOApr2026all.xlsx" in uploaded:
            return Path("WEOApr2026all.xlsx")
    except Exception:
        pass

    raise FileNotFoundError(
        "WEOApr2026all.xlsx was not found. Run locally from the repo, or upload the Excel file in Google Colab."
    )

## 4. Open the Workbook and List Sheets

`pd.ExcelFile()` opens the Excel workbook and lets us inspect available sheet names before loading the data.

In [44]:
WEO_PATH = find_weo_workbook()
print("Using workbook:", WEO_PATH)

excel_file = pd.ExcelFile(WEO_PATH)
print("Workbook sheets:", excel_file.sheet_names)

Using workbook: ../../../Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx


Workbook sheets: ['April 2026 WEO', 'Countries', 'Country Groups', 'Commodity Prices', 'Country Group Composition']


## 5. Select Relevant Indicators and Standard Column Names

The WEO workbook has many indicators. For this lab, we focus on a smaller set that is useful for macroeconomic analysis. We rename them into consistent `snake_case` columns such as `gdp_growth_pct` and `inflation_pct`.

In [45]:
# We use short, consistent snake_case names in our analysis DataFrames.
# The original WEO indicator IDs are kept only while extracting the data.
INDICATOR_LABELS = {
    "NGDP_RPCH": "gdp_growth_pct",
    "PCPIPCH": "inflation_pct",
    "LUR": "unemployment_rate",
    "BCA_NGDPD": "current_account_pct_gdp",
    "GGXWDG_NGDP": "government_debt_pct_gdp",
    "NGDPDPC": "gdp_per_capita_usd",
    "NID_NGDP": "investment_pct_gdp",
    "NGSD_NGDP": "savings_pct_gdp",
    "TX_RPCH": "export_volume_growth_pct",
    "TM_RPCH": "import_volume_growth_pct",
}

## 6. Helper Functions for Cleaning and Reshaping

The raw workbook is wide: years are separate columns. The helper functions below:

- standardize column names,
- load the useful sheets,
- convert wide year columns into tidy long format,
- create one `country_macro` DataFrame with one row per country and year.

In [46]:
def standardize_column_names(frame):
    """Convert non-year column names to snake_case for consistent analysis code."""
    renamed_columns = {}

    for column in frame.columns:
        # WEO year columns arrive as integers such as 1980, 2026, and 2031.
        # Keep them as integers because they are easier to identify and melt later.
        if isinstance(column, int):
            renamed_columns[column] = column
            continue

        clean_name = (
            str(column)
            .strip()
            .lower()
            .replace(".", "_")
            .replace(" ", "_")
            .replace("-", "_")
        )
        renamed_columns[column] = clean_name

    return frame.rename(columns=renamed_columns)


def get_year_columns(frame):
    """Return columns that represent years, for example 1980, 2024, or 2031."""
    return [column for column in frame.columns if isinstance(column, int) or str(column).isdigit()]


def load_weo_sheets(path):
    """Load and standardize the useful WEO workbook sheets."""
    countries = standardize_column_names(pd.read_excel(path, sheet_name="Countries"))
    country_groups = standardize_column_names(pd.read_excel(path, sheet_name="Country Groups"))
    commodity_prices = standardize_column_names(pd.read_excel(path, sheet_name="Commodity Prices"))
    group_composition = standardize_column_names(pd.read_excel(path, sheet_name="Country Group Composition"))

    # This sheet uses compact column names like groupname and countrycode.
    # Rename them once so the rest of the notebook can use readable snake_case names.
    group_composition = group_composition.rename(columns={
        "groupcode": "group_code",
        "groupname": "group_name",
        "groupcode_previous": "group_code_previous",
        "countrycode": "country_id",
        "countryname": "country_name",
        "countrycode_previous": "country_code_previous",
    })

    return countries, country_groups, commodity_prices, group_composition


def make_long_indicator_data(frame, indicator_ids, source_sheet):
    """Convert WEO wide year columns into a tidy row-per-country-year format."""
    year_columns = get_year_columns(frame)
    id_columns = ["country_id", "country", "indicator_id", "indicator", "unit"]

    filtered = frame.loc[frame["indicator_id"].isin(indicator_ids), id_columns + year_columns].copy()

    long_df = filtered.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name="year",
        value_name="value",
    )
    long_df["year"] = long_df["year"].astype(int)
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df["source_sheet"] = source_sheet
    return long_df


def make_country_macro_dataframe(countries, group_composition):
    """Create one country-year row with selected WEO indicators as columns."""
    country_long = make_long_indicator_data(countries, list(INDICATOR_LABELS), "Countries")

    country_macro = (
        country_long.pivot_table(
            index=["country_id", "country", "year"],
            columns="indicator_id",
            values="value",
            aggfunc="first",
        )
        .reset_index()
        .rename(columns=INDICATOR_LABELS)
    )

    # Remove the pivot column-axis name so displays do not show an old label above the header row.
    country_macro.columns.name = None

    advanced_codes = set(group_composition.loc[group_composition["group_name"].eq("Advanced Economies"), "country_id"])
    emerging_codes = set(group_composition.loc[group_composition["group_name"].eq("Emerging Market and Developing Economies"), "country_id"])
    ssa_codes = set(group_composition.loc[group_composition["group_name"].eq("Sub-Saharan Africa (SSA)"), "country_id"])

    country_macro["economic_group"] = np.select(
        [country_macro["country_id"].isin(advanced_codes), country_macro["country_id"].isin(emerging_codes)],
        ["Advanced Economies", "Emerging Market and Developing Economies"],
        default="Other",
    )
    country_macro["is_sub_saharan_africa"] = country_macro["country_id"].isin(ssa_codes)

    return country_macro, country_long

## 7. Load Sheets and Build the Analysis DataFrames

The output should show the raw row count from the `Countries` sheet and the number of country-year rows created for analysis. With the current workbook, the analysis table has about 9,398 country-year rows.

In [47]:
countries_raw, country_groups_raw, commodity_prices_raw, group_composition = load_weo_sheets(WEO_PATH)
country_macro, country_long = make_country_macro_dataframe(countries_raw, group_composition)

print("Countries sheet rows:", len(countries_raw))
print("Country-year analytical rows:", len(country_macro))
display(country_macro.head())

Countries sheet rows: 8668
Country-year analytical rows: 9398


,country_id,country,year,current_account_pct_gdp,government_debt_pct_gdp,unemployment_rate,gdp_per_capita_usd,gdp_growth_pct,savings_pct_gdp,investment_pct_gdp,inflation_pct,import_volume_growth_pct,export_volume_growth_pct,economic_group,is_sub_saharan_africa
0,ABW,"Aruba, Kingdom of the Netherlands",1986,NaN,NaN,19.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Emerging Market and Developing Economies,False
1,ABW,"Aruba, Kingdom of the Netherlands",1987,NaN,NaN,14.5,NaN,16.079,NaN,NaN,3.639,NaN,NaN,Emerging Market and Developing Economies,False
2,ABW,"Aruba, Kingdom of the Netherlands",1988,NaN,NaN,5.0,NaN,18.640,NaN,NaN,3.121,NaN,NaN,Emerging Market and Developing Economies,False
3,ABW,"Aruba, Kingdom of the Netherlands",1989,NaN,NaN,1.5,NaN,12.095,NaN,NaN,3.989,NaN,NaN,Emerging Market and Developing Economies,False
4,ABW,"Aruba, Kingdom of the Netherlands",1990,NaN,NaN,1.3,NaN,3.977,NaN,NaN,5.838,NaN,NaN,Emerging Market and Developing Economies,False


## 8. Recent-Year EDA Summary

EDA means exploratory data analysis. Here we filter to `analysis_year = 2026` and inspect the key indicators before running statistical tests.

In [48]:
analysis_year = 2026
latest = country_macro.loc[country_macro["year"].eq(analysis_year)].copy()

selected_columns = [
    "country",
    "economic_group",
    "is_sub_saharan_africa",
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_rate",
    "government_debt_pct_gdp",
    "current_account_pct_gdp",
    "gdp_per_capita_usd",
]

# Show a few rows to inspect the shape of the final DataFrame.
display(latest[selected_columns].head(10))

# describe().T gives one row per numeric column, which is easier to read.
display(latest[selected_columns[3:]].describe().T.round(2))

,country,economic_group,is_sub_saharan_africa,gdp_growth_pct,inflation_pct,unemployment_rate,government_debt_pct_gdp,current_account_pct_gdp,gdp_per_capita_usd
40,"Aruba, Kingdom of the Netherlands",Emerging Market and Developing Economies,False,2.250,1.193,4.345,60.261,6.462,42862.309
116,Angola,Emerging Market and Developing Economies,True,2.294,12.897,NaN,51.579,2.210,3754.028
168,Albania,Emerging Market and Developing Economies,False,3.409,3.376,8.700,51.540,-2.811,12492.636
199,"Andorra, Principality of",Advanced Economies,False,2.100,2.967,1.100,28.046,15.959,53474.513
251,United Arab Emirates,Emerging Market and Developing Economies,False,3.143,2.500,NaN,31.360,11.415,54214.399
303,Argentina,Emerging Market and Developing Economies,False,3.500,30.360,7.225,70.414,-0.790,14356.619
343,"Armenia, Republic of",Emerging Market and Developing Economies,False,5.300,3.620,12.800,50.594,-5.950,10409.851
395,Antigua and Barbuda,Emerging Market and Developing Economies,False,2.593,2.603,NaN,66.482,-11.768,22447.516
447,Australia,Advanced Economies,False,2.009,4.019,4.214,50.645,-2.312,75647.841
499,Austria,Advanced Economies,False,0.682,2.451,5.719,82.050,0.438,67761.236


,count,mean,std,min,25%,50%,75%,max
gdp_growth_pct,191.0,2.96,2.49,-8.65,1.95,2.91,4.07,16.17
inflation_pct,191.0,7.54,28.87,-0.44,2.50,3.35,5.53,387.40
unemployment_rate,103.0,6.59,6.79,1.00,3.86,5.10,7.31,61.33
government_debt_pct_gdp,184.0,58.24,33.57,0.52,34.94,52.46,71.66,204.35
current_account_pct_gdp,190.0,-1.84,9.78,-44.84,-5.08,-1.71,2.81,34.99
gdp_per_capita_usd,190.0,22803.84,31843.41,383.75,3230.01,9524.42,30938.84,226809.14


## 9. SciPy Descriptive Statistics

`stats.describe()` summarizes one numeric variable.

For skewness:

- close to `0` means roughly balanced,
- positive means a longer right tail,
- negative means a longer left tail.

For kurtosis in SciPy, `0` is close to a normal bell-shaped distribution. Large positive values mean more extreme tails/outliers than a normal distribution.

In [49]:
growth = latest["gdp_growth_pct"].dropna().to_numpy()
growth_stats = stats.describe(growth)

print("Number of countries:", growth_stats.nobs)
print("Minimum GDP growth:", round(growth_stats.minmax[0], 2))
print("Maximum GDP growth:", round(growth_stats.minmax[1], 2))
print("Mean GDP growth:", round(growth_stats.mean, 2))
print("Variance:", round(growth_stats.variance, 2))
print("Skewness:", round(growth_stats.skewness, 2))
print("Kurtosis:", round(growth_stats.kurtosis, 2))

Number of countries: 191
Minimum GDP growth: -8.65
Maximum GDP growth: 16.17
Mean GDP growth: 2.96
Variance: 6.19
Skewness: -0.18
Kurtosis: 7.28


### Interpreting the Descriptive Statistics

With the current workbook, 191 countries have a 2026 GDP growth value. The mean is about **2.96%**, with values ranging from about **-8.65%** to **16.17%**.

The skewness is about **-0.18**, which is close to zero, so the distribution is not strongly tilted left or right. The kurtosis is about **7.28**, which is high. That means the distribution has more extreme values than a normal distribution, so outlier review is important.

## 10. Confidence Interval

A confidence interval estimates a likely range for the true average. A 95% confidence interval means that if we repeated this sampling process many times, about 95% of intervals would contain the true mean.

The interval is in the same unit as the variable: percentage points of GDP growth.

In [50]:
mean_growth = np.mean(growth)
standard_error = stats.sem(growth)
confidence_interval = stats.t.interval(
    confidence=0.95,
    df=len(growth) - 1,
    loc=mean_growth,
    scale=standard_error,
)

print(f"{analysis_year} mean GDP growth:", round(mean_growth, 2))
print("95% confidence interval:", tuple(round(float(value), 2) for value in confidence_interval))

2026 mean GDP growth: 2.96
95% confidence interval: (2.61, 3.32)


### Interpreting the Confidence Interval

The current workbook gives a 95% interval of about **2.61% to 3.32%** for the average 2026 country-level GDP growth rate. This means the estimated average is around **2.96%**, but normal sampling uncertainty puts the likely average in that range.

## 11. Correlation: Inflation and GDP Growth

Correlation ranges from `-1` to `1`:

- `1` means a perfect positive relationship,
- `0` means no linear relationship,
- `-1` means a perfect negative relationship.

The p-value ranges from `0` to `1`. A p-value below `0.05` is commonly treated as statistically significant evidence against "no relationship".

In [55]:
corr_df = latest[["gdp_growth_pct", "inflation_pct"]].dropna()

# Pearson checks a linear relationship.
pearson_corr, pearson_p = stats.pearsonr(corr_df["inflation_pct"], corr_df["gdp_growth_pct"])

# Spearman checks whether the ranking of one variable moves with the ranking of the other.
spearman_corr, spearman_p = stats.spearmanr(corr_df["inflation_pct"], corr_df["gdp_growth_pct"])

print("Pearson correlation:", round(pearson_corr, 3), "p-value:", round(pearson_p, 4))
print("Spearman correlation:", round(spearman_corr, 3), "p-value:", round(spearman_p, 4))

Pearson correlation: -0.025 p-value: 0.7355
Spearman correlation: 0.115 p-value: 0.1139


### Interpreting the Correlation Output

The current results show Pearson correlation around **-0.025** with p-value around **0.736**, and Spearman correlation around **0.115** with p-value around **0.114**.

Both p-values are above `0.05`, so this notebook does **not** find statistically significant evidence of a simple inflation-growth relationship in this 2026 cross-section. This does not prove no economic relationship exists; it only means this simple two-variable check is weak for this year.

## 12. Outlier Review with Z-Scores

A z-score tells how many standard deviations a value is from the mean.

A common beginner rule is:

- `abs(z_score) <= 2`: usually not extreme,
- `abs(z_score) > 2`: review as a possible outlier,
- `abs(z_score) > 3`: very unusual compared with the rest of the data.

In [52]:
latest["growth_z_score"] = stats.zscore(latest["gdp_growth_pct"], nan_policy="omit")
latest["growth_outlier_flag"] = latest["growth_z_score"].abs() > 2

outlier_columns = ["country", "economic_group", "gdp_growth_pct", "growth_z_score"]
outliers = latest.loc[latest["growth_outlier_flag"], outlier_columns].sort_values("growth_z_score")
display(outliers)

,country,economic_group,gdp_growth_pct,growth_z_score
7102,Qatar,Emerging Market and Developing Economies,-8.646,-4.677116
4023,Iraq,Emerging Market and Developing Economies,-6.771,-3.921722
3992,"Iran, Islamic Republic of",Emerging Market and Developing Economies,-6.070,-3.639306
1137,Bolivia,Emerging Market and Developing Economies,-3.285,-2.517295
3327,"Equatorial Guinea, Republic of",Emerging Market and Developing Economies,-2.686,-2.275972
3171,Guinea,Emerging Market and Developing Economies,8.677,2.301914
2732,"Ethiopia, The Federal Democratic Republic of",Emerging Market and Developing Economies,9.204,2.514230
3535,Guyana,Emerging Market and Developing Economies,16.168,5.319862


### Interpreting the Outliers

The current workbook flags countries such as **Qatar**, **Iraq**, **Iran**, and **Guyana** as growth outliers for 2026. These rows should not be deleted automatically. Outliers may reflect real economic events, data revisions, or forecast assumptions. The correct next step is to review them in context.

## 13. Hypothesis Test: Economic Group Difference

This t-test compares average 2026 GDP growth between advanced economies and emerging/developing economies.

The p-value is between `0` and `1`. Very small values may appear in scientific notation. For example, `5.395e-07` means `0.0000005395`, not `5.395`.

In [53]:
advanced_growth = latest.loc[latest["economic_group"].eq("Advanced Economies"), "gdp_growth_pct"].dropna()
emerging_growth = latest.loc[latest["economic_group"].eq("Emerging Market and Developing Economies"), "gdp_growth_pct"].dropna()

t_stat, p_value = stats.ttest_ind(advanced_growth, emerging_growth, equal_var=False)

print("Advanced economies mean:", round(advanced_growth.mean(), 2))
print("Emerging/developing economies mean:", round(emerging_growth.mean(), 2))
print("t-statistic:", round(t_stat, 3))
print("p-value:", p_value)

if p_value < 0.05:
    print("Interpretation: the group difference is statistically significant at the 5% level.")
else:
    print("Interpretation: the group difference is not statistically significant at the 5% level.")

Advanced economies mean: 1.85
Emerging/developing economies mean: 3.29
t-statistic: -5.209
p-value: 5.395585152092368e-07
Interpretation: the group difference is statistically significant at the 5% level.


### Interpreting the T-Test

The current output shows advanced economies averaging about **1.85%** growth and emerging/developing economies averaging about **3.29%** growth. The p-value is about **5.40e-07**, which is far below `0.05`.

That means the observed group difference is statistically significant in this workbook. In plain language: the two groups show meaningfully different average projected GDP growth for 2026.

## 14. Export Statistical Outputs

Exports make notebook results available for reports or later labs.

In [54]:
summary_path = OUTPUT_DIR / "weo_scipy_2026_summary.csv"
outliers_path = OUTPUT_DIR / "weo_scipy_growth_outliers.csv"

latest[selected_columns[3:]].describe().T.round(2).to_csv(summary_path)
outliers.to_csv(outliers_path, index=False)

print("Saved:", summary_path)
print("Saved:", outliers_path)

Saved: outputs/weo_scipy_2026_summary.csv
Saved: outputs/weo_scipy_growth_outliers.csv
